In [1]:
import os, threading, http.server, socketserver
from google.colab import output

os.makedirs("/content/blog", exist_ok=True)

HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>React Blog Management</title>
<style>
:root{--p:#4f46e5;--pd:#4338ca;--bg:#f5f6fa;--t:#1f2937;--m:#6b7280;--d:#dc2626;--b:#e5e7eb}
*{box-sizing:border-box}
body{margin:0;font-family:"Segoe UI",system-ui,sans-serif;background:var(--bg);color:var(--t);line-height:1.6}
a{color:var(--p);text-decoration:none;cursor:pointer}a:hover{text-decoration:underline}
.nav{background:var(--p);padding:12px 20px;display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;align-items:center}
.nav a,.nav button{color:#e0e7ff;margin-left:16px}.nav .brand{color:#fff;font-size:1.3rem;font-weight:700;margin:0}
.wrap{max-width:1000px;margin:0 auto;padding:24px 16px 50px}
.hero{text-align:center;padding:20px 0}.hero p,.meta{color:var(--m);font-size:.88rem;margin:4px 0}
.bar{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px}.bar input{flex:1;min-width:200px}
input,select,textarea{padding:10px 12px;border:1px solid var(--b);border-radius:8px;font:inherit;width:100%;background:#fff}
.bar select{width:auto}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(280px,1fr));gap:20px}
.card{background:#fff;border-radius:12px;overflow:hidden;box-shadow:0 1px 4px rgba(0,0,0,.1)}
.card img{width:100%;height:150px;object-fit:cover;background:#e5e7eb}.cb{padding:14px 16px}
.badge{background:#eef2ff;color:var(--pd);padding:2px 10px;border-radius:99px;font-size:.75rem;font-weight:600}
.tag{font-size:.78rem;color:var(--m);background:#f3f4f6;padding:1px 8px;border-radius:6px;margin-right:6px}
.btn{background:var(--p);color:#fff;border:0;padding:9px 18px;border-radius:8px;cursor:pointer;font:inherit}
.btn:hover{background:var(--pd)}.btn:disabled{opacity:.5}.btn.s{background:#e5e7eb;color:var(--t)}
.lb{background:none;border:0;cursor:pointer;font:inherit;color:var(--p);padding:0}.lb.d{color:var(--d)}
.box{background:#fff;border-radius:12px;padding:22px;box-shadow:0 1px 4px rgba(0,0,0,.1)}
.form{display:flex;flex-direction:column;gap:8px}.form label{font-weight:600;font-size:.9rem}
.narrow{max-width:380px;margin:30px auto}.err{color:var(--d);font-size:.85rem;margin:0}
.cm{background:#f9fafb;border-radius:8px;padding:8px 14px;margin-bottom:10px}.cm p{margin:4px 0}
.row{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0;align-items:center}
.stat{background:#fff;padding:14px 28px;border-radius:12px;box-shadow:0 1px 4px rgba(0,0,0,.1);text-align:center}
.stat b{display:block;font-size:1.8rem;color:var(--p)}
table{width:100%;border-collapse:collapse;background:#fff;border-radius:10px;overflow:hidden}
th,td{text-align:left;padding:10px 12px;border-bottom:1px solid var(--b);font-size:.92rem}th{background:#eef2ff}
.foot{text-align:center;color:var(--m);padding:20px;font-size:.85rem}
</style>
</head>
<body>
<div id="root"></div>
<script src="https://unpkg.com/react@18.3.1/umd/react.development.js"></script>
<script src="https://unpkg.com/react-dom@18.3.1/umd/react-dom.development.js"></script>
<script src="https://unpkg.com/@babel/standalone@7.25.6/babel.min.js"></script>
<script type="text/babel">
const { useState, useEffect, useMemo } = React;

/* ------------------------- DATA ------------------------- */
const CATEGORIES = ["Technology", "Web Development", "Lifestyle", "Education", "Travel"];
const ADMIN = { username: "admin", password: "admin123" };
const SEED = [
  { id: 1, title: "Getting Started with React Hooks", category: "Web Development", author: "Priya", date: "2026-09-02",
    tags: ["react", "hooks", "javascript"], image: "https://picsum.photos/seed/react-hooks/800/400", likes: 12,
    content: "Hooks let you use state and other React features without writing a class.\n\nThe most common hooks are useState for local state and useEffect for side effects such as fetching data. Custom hooks allow you to share logic between components.",
    comments: [{ id: 1, name: "Arun", text: "Very clear explanation, thanks!", date: "2026-09-03" }, { id: 2, name: "Meena", text: "Please write about useReducer next.", date: "2026-09-04" }] },
  { id: 2, title: "Why Every Student Should Learn Git", category: "Education", author: "Karthik", date: "2026-09-10",
    tags: ["git", "github", "version-control"], image: "https://picsum.photos/seed/git-learn/800/400", likes: 8,
    content: "Git is a version control system that tracks changes in your source code.\n\nUsing Git you can experiment safely with branches, collaborate through pull requests and return to an earlier working version.",
    comments: [{ id: 1, name: "Divya", text: "Branching finally makes sense now.", date: "2026-09-11" }] },
  { id: 3, title: "Top 5 Places to Visit in South India", category: "Travel", author: "Lakshmi", date: "2026-09-18",
    tags: ["travel", "india", "holiday"], image: "https://picsum.photos/seed/south-india/800/400", likes: 21,
    content: "South India offers beaches, hill stations, temples and delicious food.\n\nMunnar, Hampi, Pondicherry, Kochi backwaters and Ooty are five destinations for every traveller's list.",
    comments: [] },
  { id: 4, title: "Introduction to Artificial Intelligence", category: "Technology", author: "Priya", date: "2026-10-01",
    tags: ["ai", "machine-learning", "future"], image: "https://picsum.photos/seed/ai-intro/800/400", likes: 15,
    content: "Artificial Intelligence is the ability of a machine to perform tasks that normally need human intelligence.\n\nExamples include voice assistants, recommendation engines and self-driving vehicles.",
    comments: [{ id: 1, name: "Surya", text: "Great overview for beginners.", date: "2026-10-02" }] },
  { id: 5, title: "Simple Habits for a Healthy Daily Routine", category: "Lifestyle", author: "Lakshmi", date: "2026-10-12",
    tags: ["health", "routine", "habits"], image: "https://picsum.photos/seed/healthy-routine/800/400", likes: 6,
    content: "Small habits create big changes. Wake up at a fixed time, drink enough water, take short breaks from the screen and sleep for seven to eight hours.\n\nTracking your habits in a notebook helps you stay consistent.",
    comments: [] },
];
const load = (k, f) => { try { const r = localStorage.getItem(k); return r ? JSON.parse(r) : f; } catch { return f; } };
const save = (k, v) => { try { localStorage.setItem(k, JSON.stringify(v)); } catch {} };
const today = () => new Date().toISOString().slice(0, 10);
const go = (p) => { location.hash = p; };
const hide = (e) => { e.currentTarget.style.display = "none"; };

/* ------------------------- ROUTER (hash based) ------------------------- */
function useRoute() {
  const [r, setR] = useState(location.hash.slice(1) || "/");
  useEffect(() => {
    const f = () => setR(location.hash.slice(1) || "/");
    window.addEventListener("hashchange", f);
    return () => window.removeEventListener("hashchange", f);
  }, []);
  return r;
}

/* ------------------------- PAGES ------------------------- */
function Home({ posts }) {
  const [q, setQ] = useState(""), [cat, setCat] = useState("All"), [sort, setSort] = useState("new"), [page, setPage] = useState(1);
  const list = useMemo(() => {
    const s = q.toLowerCase();
    const f = posts.filter(p => (cat === "All" || p.category === cat) &&
      (p.title.toLowerCase().includes(s) || p.content.toLowerCase().includes(s) || p.tags.some(t => t.includes(s))));
    return f.sort((a, b) => sort === "new" ? b.date.localeCompare(a.date) : sort === "old" ? a.date.localeCompare(b.date) : b.likes - a.likes);
  }, [posts, q, cat, sort]);
  const pages = Math.max(1, Math.ceil(list.length / 4)), cur = Math.min(page, pages);
  return (<>
    <div className="hero"><h1>Welcome to ReactBlog</h1><p>Read, search and share articles on technology, education, travel and lifestyle.</p></div>
    <div className="bar">
      <input placeholder="Search by title, content or tag…" value={q} onChange={e => { setQ(e.target.value); setPage(1); }} />
      <select value={cat} onChange={e => { setCat(e.target.value); setPage(1); }}><option>All</option>{CATEGORIES.map(c => <option key={c}>{c}</option>)}</select>
      <select value={sort} onChange={e => setSort(e.target.value)}><option value="new">Newest first</option><option value="old">Oldest first</option><option value="liked">Most liked</option></select>
    </div>
    {list.length === 0 ? <p className="meta" style={{ textAlign: "center" }}>No posts found.</p> :
      <div className="grid">{list.slice((cur - 1) * 4, cur * 4).map(p => (
        <div className="card" key={p.id}>
          {p.image && <img src={p.image} alt="" onError={hide} />}
          <div className="cb"><span className="badge">{p.category}</span>
            <h3 style={{ margin: "8px 0 2px" }}><a onClick={() => go("/post/" + p.id)}>{p.title}</a></h3>
            <p className="meta">By {p.author} · {p.date}</p>
            <p>{p.content.length > 110 ? p.content.slice(0, 110) + "…" : p.content}</p>
            <div>{p.tags.map(t => <span className="tag" key={t}>#{t}</span>)}</div>
            <p className="meta">♥ {p.likes} · 💬 {p.comments.length}</p></div>
        </div>))}</div>}
    {pages > 1 && <div className="row" style={{ justifyContent: "center" }}>
      <button className="btn" disabled={cur === 1} onClick={() => setPage(cur - 1)}>Prev</button>
      <span>Page {cur} of {pages}</span>
      <button className="btn" disabled={cur === pages} onClick={() => setPage(cur + 1)}>Next</button></div>}
  </>);
}

function Detail({ id, posts, user, update }) {
  const p = posts.find(x => x.id === Number(id));
  const [name, setName] = useState(""), [text, setText] = useState("");
  if (!p) return <p>Post not found. <a onClick={() => go("/")}>Back</a></p>;
  const addComment = (e) => {
    e.preventDefault();
    if (!name.trim() || !text.trim()) return;
    const cid = p.comments.length ? Math.max(...p.comments.map(c => c.id)) + 1 : 1;
    update(p.id, { comments: [...p.comments, { id: cid, name: name.trim(), text: text.trim(), date: today() }] });
    setName(""); setText("");
  };
  return (<div className="box">
    <a onClick={() => go("/")}>← Back</a> <span className="badge">{p.category}</span>
    <h1>{p.title}</h1><p className="meta">By {p.author} · {p.date}</p>
    {p.image && <img src={p.image} alt="" onError={hide} style={{ width: "100%", maxHeight: 360, objectFit: "cover", borderRadius: 10 }} />}
    {p.content.split("\n\n").map((t, i) => <p key={i}>{t}</p>)}
    <div>{p.tags.map(t => <span className="tag" key={t}>#{t}</span>)}</div>
    <div className="row">
      <button className="btn" onClick={() => update(p.id, { likes: p.likes + 1 })}>♥ Like ({p.likes})</button>
      {user && <button className="btn s" onClick={() => go("/edit/" + p.id)}>Edit</button>}
    </div>
    <h3>Comments ({p.comments.length})</h3>
    {p.comments.length === 0 && <p className="meta">Be the first to comment.</p>}
    {p.comments.map(c => (<div className="cm" key={c.id}><b>{c.name}</b> <span className="meta">· {c.date}</span><p>{c.text}</p>
      {user && <button className="lb d" onClick={() => update(p.id, { comments: p.comments.filter(x => x.id !== c.id) })}>Delete</button>}</div>))}
    <form className="form" onSubmit={addComment}>
      <input placeholder="Your name" value={name} onChange={e => setName(e.target.value)} />
      <textarea rows="3" placeholder="Write a comment…" value={text} onChange={e => setText(e.target.value)} />
      <div><button className="btn" type="submit">Post Comment</button></div>
    </form>
  </div>);
}

function Login({ login }) {
  const [u, setU] = useState(""), [pw, setPw] = useState(""), [err, setErr] = useState("");
  const submit = (e) => { e.preventDefault(); if (login(u, pw)) go("/dashboard"); else setErr("Invalid username or password."); };
  return (<form className="box form narrow" onSubmit={submit}>
    <h2>Admin Login</h2>
    <input placeholder="Username" value={u} onChange={e => setU(e.target.value)} />
    <input type="password" placeholder="Password" value={pw} onChange={e => setPw(e.target.value)} />
    {err && <p className="err">{err}</p>}
    <div><button className="btn" type="submit">Login</button></div>
    <p className="meta">Demo credentials: admin / admin123</p>
  </form>);
}

function Dashboard({ posts, remove, reset }) {
  return (<>
    <h2>Admin Dashboard</h2>
    <div className="row">
      <div className="stat"><b>{posts.length}</b>Posts</div>
      <div className="stat"><b>{posts.reduce((s, p) => s + p.likes, 0)}</b>Likes</div>
      <div className="stat"><b>{posts.reduce((s, p) => s + p.comments.length, 0)}</b>Comments</div>
    </div>
    <div className="row">
      <button className="btn" onClick={() => go("/new")}>+ New Post</button>
      <button className="btn s" onClick={() => window.confirm("Restore sample data?") && reset()}>Reset sample data</button>
    </div>
    <table><thead><tr><th>Title</th><th>Category</th><th>Date</th><th>Likes</th><th>Comments</th><th>Actions</th></tr></thead>
      <tbody>{posts.map(p => (<tr key={p.id}>
        <td><a onClick={() => go("/post/" + p.id)}>{p.title}</a></td><td>{p.category}</td><td>{p.date}</td><td>{p.likes}</td><td>{p.comments.length}</td>
        <td><a onClick={() => go("/edit/" + p.id)}>Edit</a> · <button className="lb d" onClick={() => window.confirm('Delete "' + p.title + '"?') && remove(p.id)}>Delete</button></td>
      </tr>))}</tbody></table>
  </>);
}

function Editor({ id, posts, user, add, update }) {
  const ex = id ? posts.find(x => x.id === Number(id)) : null;
  const [f, setF] = useState({
    title: ex ? ex.title : "", category: ex ? ex.category : CATEGORIES[0], author: ex ? ex.author : user.username,
    image: ex ? ex.image : "", tags: ex ? ex.tags.join(", ") : "", content: ex ? ex.content : "" });
  const [er, setEr] = useState({});
  const ch = (e) => setF({ ...f, [e.target.name]: e.target.value });
  const submit = (e) => {
    e.preventDefault();
    const x = {};
    if (f.title.trim().length < 5) x.title = "Title must be at least 5 characters.";
    if (!f.author.trim()) x.author = "Author is required.";
    if (f.content.trim().length < 30) x.content = "Content must be at least 30 characters.";
    setEr(x);
    if (Object.keys(x).length) return;
    const data = { title: f.title.trim(), category: f.category, author: f.author.trim(), image: f.image.trim(), content: f.content.trim(),
      tags: f.tags.split(",").map(t => t.trim().toLowerCase()).filter(Boolean) };
    if (ex) { update(ex.id, data); go("/post/" + ex.id); } else go("/post/" + add(data));
  };
  if (id && !ex) return <p>Post not found.</p>;
  return (<form className="box form" onSubmit={submit}>
    <h2>{ex ? "Edit Post" : "Create New Post"}</h2>
    <label>Title</label><input name="title" value={f.title} onChange={ch} />{er.title && <p className="err">{er.title}</p>}
    <label>Category</label><select name="category" value={f.category} onChange={ch}>{CATEGORIES.map(c => <option key={c}>{c}</option>)}</select>
    <label>Author</label><input name="author" value={f.author} onChange={ch} />{er.author && <p className="err">{er.author}</p>}
    <label>Image URL (optional)</label><input name="image" value={f.image} onChange={ch} />
    <label>Tags (comma separated)</label><input name="tags" value={f.tags} onChange={ch} />
    <label>Content (blank line = new paragraph)</label><textarea name="content" rows="8" value={f.content} onChange={ch} />{er.content && <p className="err">{er.content}</p>}
    <div className="row"><button className="btn" type="submit">{ex ? "Update" : "Publish"}</button><button className="btn s" type="button" onClick={() => history.back()}>Cancel</button></div>
  </form>);
}

/* ------------------------- APP (state + routes) ------------------------- */
function App() {
  const route = useRoute();
  const [posts, setPosts] = useState(() => load("blog_posts_v1", SEED));
  const [user, setUser] = useState(() => load("blog_auth_v1", null));
  useEffect(() => save("blog_posts_v1", posts), [posts]);
  useEffect(() => save("blog_auth_v1", user), [user]);

  const login = (u, p) => { if (u === ADMIN.username && p === ADMIN.password) { setUser({ username: u }); return true; } return false; };
  const logout = () => { setUser(null); go("/"); };
  const update = (id, data) => setPosts(ps => ps.map(p => p.id === id ? { ...p, ...data } : p));
  const remove = (id) => setPosts(ps => ps.filter(p => p.id !== id));
  const add = (data) => {
    const id = posts.length ? Math.max(...posts.map(p => p.id)) + 1 : 1;
    setPosts([{ ...data, id, date: today(), likes: 0, comments: [] }, ...posts]);
    return id;
  };

  const [, a, b] = route.split("/");               // "/post/3" -> ["", "post", "3"]
  const needLogin = ["dashboard", "new", "edit"].includes(a);
  let page;
  if (needLogin && !user) page = <Login login={login} />;
  else if (a === "post") page = <Detail id={b} posts={posts} user={user} update={update} />;
  else if (a === "login") page = <Login login={login} />;
  else if (a === "dashboard") page = <Dashboard posts={posts} remove={remove} reset={() => setPosts(SEED)} />;
  else if (a === "new") page = <Editor key="new" posts={posts} user={user} add={add} update={update} />;
  else if (a === "edit") page = <Editor key={b} id={b} posts={posts} user={user} add={add} update={update} />;
  else page = <Home posts={posts} />;

  return (<>
    <div className="nav"><a className="brand" onClick={() => go("/")}>✍ ReactBlog</a>
      <div><a onClick={() => go("/")}>Home</a>
        {user ? <><a onClick={() => go("/dashboard")}>Dashboard</a><a onClick={() => go("/new")}>New Post</a><button className="lb" onClick={logout}>Logout ({user.username})</button></>
              : <a onClick={() => go("/login")}>Admin Login</a>}</div></div>
    <div className="wrap">{page}</div>
    <div className="foot">React Blog Management © 2026 · Modern Web Programming Project</div>
  </>);
}
ReactDOM.createRoot(document.getElementById("root")).render(<App />);
</script>
</body>
</html>
'''

with open("/content/blog/index.html", "w", encoding="utf-8") as f:
    f.write(HTML)

class Handler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory="/content/blog", **k)
    def log_message(self, *a):
        pass

socketserver.TCPServer.allow_reuse_address = True
server = socketserver.ThreadingTCPServer(("", 8000), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()

print("✔ Blog is running. Admin login: admin / admin123")
output.serve_kernel_port_as_iframe(8000, height="800")

✔ Blog is running. Admin login: admin / admin123


<IPython.core.display.Javascript object>